In [ ]:
import torch, torchvision
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import requests
from io import BytesIO

# Función para cargar y preprocesar
def load_image(url, max_size=400):
    response = requests.get(url)
    image = Image.open(BytesIO(response.content)).convert('RGB')
    size = min(max_size, max(image.size))
    transform = transforms.Compose([
        transforms.Resize(size),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x[:3, :, :].unsqueeze(0))
    ])
    return transform(image)

content_url = "https://pytorch.org/tutorials/_static/img/neural-style/picasso.jpg"
style_url = "https://pytorch.org/tutorials/_static/img/neural-style/dancing.jpg"

content = load_image(content_url)
style = load_image(style_url)

plt.figure(figsize=(8,4))
plt.subplot(1,2,1); plt.imshow(content.squeeze().permute(1,2,0)); plt.title("Contenido")
plt.subplot(1,2,2); plt.imshow(style.squeeze().permute(1,2,0)); plt.title("Estilo")
plt.show()


In [ ]:
vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features
for param in vgg.parameters():
    param.requires_grad_(False)
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vgg.to(device)
content, style = content.to(device), style.to(device)

In [ ]:
def get_features(image, model, layers=None):
    if layers is None:
        layers = {'0': 'conv1_1',
                  '5': 'conv2_1',
                  '10': 'conv3_1',
                  '19': 'conv4_1',
                  '21': 'conv4_2',  # contenido
                  '28': 'conv5_1'}
    features = {}
    x = image
    for name, layer in model._modules.items():
        x = layer(x)
        if name in layers:
            features[layers[name]] = x
    return features

def gram_matrix(tensor):
    _, d, h, w = tensor.size()
    tensor = tensor.view(d, h * w)
    gram = torch.mm(tensor, tensor.t())
    return gram / (d * h * w)


In [ ]:
content_features = get_features(content, vgg)
style_features = get_features(style, vgg)
style_grams = {layer: gram_matrix(style_features[layer]) for layer in style_features}

target = content.clone().requires_grad_(True).to(device)

style_weights = {'conv1_1': 1.,
                 'conv2_1': 0.75,
                 'conv3_1': 0.2,
                 'conv4_1': 0.2,
                 'conv5_1': 0.2}
content_weight = 1e4  # α
style_weight = 1e2    # β


In [ ]:
optimizer = torch.optim.Adam([target], lr=0.003)

for i in range(1, 201):
    target_features = get_features(target, vgg)
    content_loss = torch.mean((target_features['conv4_2'] - content_features['conv4_2'])**2)
    style_loss = 0
    for layer in style_weights:
        target_f = target_features[layer]
        target_gram = gram_matrix(target_f)
        style_gram = style_grams[layer]
        layer_loss = style_weights[layer] * torch.mean((target_gram - style_gram)**2)
        _, d, h, w = target_f.shape
        style_loss += layer_loss / (d * h * w)
    total_loss = content_weight * content_loss + style_weight * style_loss
    
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    
    if i % 50 == 0:
        print(f"Iter {i}, Total loss: {total_loss.item():.2f}")


In [ ]:
final_img = target.cpu().clone().squeeze()
final_img = final_img.detach().permute(1,2,0)
plt.imshow(final_img)
plt.title("Imagen generada (contenido + estilo)")
plt.show()